# Week 7 — The Price is Right (SamuelAdebodun)

Evaluate a **Llama 3.2 3B** model for product price prediction.

- **Base model:** `meta-llama/Llama-3.2-3B`
- **Dataset:** Course prompt/completion data from HuggingFace (`ed-donner/items_prompts_lite`).
- **Evaluation:** Course `util.py` (error report, scatter, trend chart).

Run training (Days 1–4) in Colab, then set `USE_PEFT = True` and `RUN_NAME` to your run and run this notebook to evaluate. With `USE_PEFT = False` you evaluate the base model only (baseline).

In [ ]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
import os
import re
from tqdm import tqdm
from huggingface_hub import login
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from datasets import load_dataset
from peft import PeftModel
from util import evaluate

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "SamuelAdebodun"

LITE_MODE = True
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

USE_PEFT = False
RUN_NAME = "2026-03-08-lite"
REVISION = None

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8 if torch.cuda.is_available() else False

In [ ]:
if IN_COLAB:
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.getenv("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset["test"]
print(f"Test size: {len(test)}")

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

if USE_PEFT:
    if REVISION:
        model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
    else:
        model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
else:
    model = base_model

print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
set_seed(42)
evaluate(model_predict, test)